In [36]:
# Albert Medina Familia
# 22-EISN-2-025

import os
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from PIL import Image

In [37]:
# Albert Medina Familia
# 22-EISN-2-025

dataset_path = "Dataset"
classes = os.listdir(dataset_path)

In [38]:
# Albert Medina Familia
# 22-EISN-2-025

classes = ["Bracelet", "Chains", "Glasses", "Phone", "Rings", "Watches"]

In [39]:
# Albert Medina Familia
# 22-EISN-2-025

data = []
for cls in classes:
    class_path = os.path.join(dataset_path, cls)
    images = os.listdir(class_path)
    for img in images:
        img_path = os.path.join(class_path, img)
        data.append((img_path, cls))

In [40]:
# Albert Medina Familia
# 22-EISN-2-025

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [41]:
# Albert Medina Familia
# 22-EISN-2-025

from torchvision.datasets import ImageFolder

dataset = ImageFolder(root=dataset_path, transform=transform)
data_loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [42]:
# Albert Medina Familia
# 22-EISN-2-025

from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT 
model = resnet18(weights=weights)

for param in model.parameters():
    param.requires_grad = False

In [43]:
# Albert Medina Familia
# 22-EISN-2-025

import torch.nn as nn

num_classes = len(classes)
model.fc = nn.Sequential(
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, num_classes)
)

In [44]:
# Albert Medina Familia
# 22-EISN-2-025

from torch.utils.data.dataset import random_split

train_size = int(0.8 * len(data_loader.dataset))
test_size = len(data_loader.dataset) - train_size
train_dataset, test_dataset = random_split(data_loader.dataset, [train_size, test_size])

In [45]:
# Albert Medina Familia
# 22-EISN-2-025

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

In [46]:
# Albert Medina Familia
# 22-EISN-2-025

dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dispositivo

device(type='cpu')

In [47]:
# Albert Medina Familia
# 22-EISN-2-025

lr = 1e-3

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

n_epochs = 10

In [48]:
# Albert Medina Familia
# 22-EISN-2-025

perdidastrain = []
perdidasval = []
acuraciatrain = []
acuraciaval = []
error_ratetrain = []
error_rateval = []

In [49]:
# Albert Medina Familia
# 22-EISN-2-025

from tqdm import tqdm

for epoch in range(n_epochs):
    model.train()
    perdida=0
    aciertos=0
    errores=0
    
    for images, labels in tqdm(train_loader):
        
        images, labels = images.to(dispositivo), labels.to(dispositivo)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        perdida+=loss.item()
        _, preds = torch.max(outputs, 1)
        aciertos += (preds == labels).sum().item()
        errores += (preds != labels).sum().item()
        
    perdidastrain.append(perdida/len(train_loader))
    acuraciatrain.append(aciertos/len(train_loader.dataset))
    error_ratetrain.append(errores/len(train_loader.dataset))

    model.eval()
    perdida=0
    aciertos=0
    errores=0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(dispositivo), labels.to(dispositivo)
            outputs = model(images)
            loss = criterion(outputs, labels)
            perdida+=loss.item()
            _, preds = torch.max(outputs, 1)
            aciertos += (preds == labels).sum().item()
            errores += (preds != labels).sum().item()
    perdidasval.append(perdida/len(test_loader))
    acuraciaval.append(aciertos/len(test_loader.dataset))
    error_rateval.append(errores/len(test_loader.dataset))
    
    print(f'Epoch {epoch+1}/{n_epochs}, train loss: {perdidastrain[-1]:.4f}, train accuracy: {acuraciatrain[-1]:.4f}, train error rate: {error_ratetrain[-1]:.4f}, val loss: {perdidasval[-1]:.4f}, val accuracy: {acuraciaval[-1]:.4f}, val error rate: {error_rateval[-1]:.4f}') 


  0%|          | 0/105 [00:00<?, ?it/s]

100%|██████████| 105/105 [06:32<00:00,  3.74s/it]


Epoch 1/10, train loss: 0.5548, train accuracy: 0.8170, train error rate: 0.1830, val loss: 0.2948, val accuracy: 0.9032, val error rate: 0.0968


100%|██████████| 105/105 [07:40<00:00,  4.39s/it]


Epoch 2/10, train loss: 0.3286, train accuracy: 0.8889, train error rate: 0.1111, val loss: 0.2630, val accuracy: 0.9050, val error rate: 0.0950


...

Epoch 8/10, train loss: 0.2167, train accuracy: 0.9227, train error rate: 0.0773, val loss: 0.2206, val accuracy: 0.9241, val error rate: 0.0759


100%|██████████| 105/105 [06:54<00:00,  3.94s/it]


Epoch 9/10, train loss: 0.2144, train accuracy: 0.9226, train error rate: 0.0774, val loss: 0.2293, val accuracy: 0.9235, val error rate: 0.0765


100%|██████████| 105/105 [06:52<00:00,  3.93s/it]


Epoch 10/10, train loss: 0.1953, train accuracy: 0.9323, train error rate: 0.0677, val loss: 0.2338, val accuracy: 0.9235, val error rate: 0.0765


In [50]:
# Albert Medina Familia
# 22-EISN-2-025

import numpy as np

with torch.no_grad():
    model.eval()
    preds = []
    targets = []
    for images, labels in test_loader:
        images, labels = images.to(dispositivo), labels.to(dispositivo)
        outputs = model(images)
        _, pred = torch.max(outputs, 1)
        preds.append(pred.cpu().numpy())
        targets.append(labels.cpu().numpy())
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

In [51]:
# Albert Medina Familia
# 22-EISN-2-025

from safetensors.torch import   save_model, load_model
save_model(model, 'detector_accesorios.safetensors')

In [52]:
# Albert Medina Familia
# 22-EISN-2-025

import torch
from torchvision import transforms
import gradio as gr
from PIL import Image
from safetensors.torch import load_model

clothes = ["Bracelet", "Chains", "Glasses", "Phone", "Rings", "Watches"]

# Cargar el modelo previamente entrenado
model = resnet18(weights=weights)
num_classes = len(clothes)
model.fc = nn.Sequential(
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, num_classes)
)
load_model(model, 'detector_accesorios.safetensors')

# Función de predicción
def predict(image):
    
    image = image['composite'].convert('RGB')
    image = image.resize((224, 224))
    image = transforms.ToTensor()(image)
    image = image.unsqueeze(0)

    # Realizar la predicción
    with torch.no_grad():
        model.eval()
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

        # Retornar la etiqueta predicha
        return clothes[pred.item()]

# Interfaz de usuario Gradio
gr.Interface(fn=predict,inputs=gr.ImageEditor(type='pil', crop_size="1:1", ) ,outputs="text").launch(share=True)

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://90615b7aaecf005683.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
